# Agregacao de dados — Apresentacao ao Presidente (TCE-RN)

Consolida os tres temas da apresentacao e gera os graficos (`assets/*.png`) e os
numeros (`numeros.json`) usados no deck Marp `scripts/apresentacao/apresentacao_presidente_tce.md`.

1. **Arrecadacao do FRAP** — `BdDIP.FRAPLancamento` (creditos por ano e forma de pagamento) + piloto Pix/Cartao (`scripts/docs/info_pix_cartao_042026.csv`).
2. **Caso Nereu** — le `scripts/analise/docs/debitos_nereu_planilha_final.xlsx` (ja consolidado).
3. **Acervo do CGAD** — `BdDIP.Obrigacao` / `Recomendacao` / `ClassificacaoDecisao` (volume, por orgao, cumprimento, evolucao).

> Fonte das tabelas FRAP* e do CGAD: banco **BdDIP**. Requer rede corporativa (MSSQL).

In [1]:
import json
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from sqlalchemy import text
from ccd.db import get_connection

REPO = Path.cwd()
while not (REPO / "ccd").is_dir() and REPO != REPO.parent:
    REPO = REPO.parent
ASSETS = REPO / "scripts" / "apresentacao" / "assets"
ASSETS.mkdir(parents=True, exist_ok=True)
DOCS = REPO / "scripts" / "analise" / "docs"

# Paleta institucional
AZUL, TEAL, OURO, CINZA, VERMELHO, VERDE = "#1f4e79", "#2a9d8f", "#e9c46a", "#8d99ae", "#c1121f", "#2a9d8f"
plt.rcParams.update({
    "figure.dpi": 130, "savefig.dpi": 130, "font.size": 12,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 14, "axes.titleweight": "bold", "figure.autolayout": True,
})

def fmt_brl(v, casas=2):
    s = f"{v:,.{casas}f}".replace(",", "#").replace(".", ",").replace("#", ".")
    return "R$ " + s

def fmt_mi(v):
    return f"R$ {v/1_000_000:.2f}M".replace(".", ",")

def salvar(fig, nome):
    p = ASSETS / nome
    fig.savefig(p, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print("salvo:", p.relative_to(REPO))

NUM = {}  # numeros consolidados -> numeros.json
eng = get_connection(db="BdDIP")
print("conectado ao BdDIP")

conectado ao BdDIP


## 1. Arrecadacao do FRAP

In [2]:
# Creditos reais (entradas) por ano e categoria. Exclui cat 4 (aplicacao/resgate) e 5 (saldo informativo).
q_frap = text('''
    SELECT YEAR(l.DtMovimento) ano, cat.Descricao categoria, COUNT(*) qtd, SUM(l.Valor) total
    FROM FRAPLancamento l JOIN FRAPCategoria cat ON cat.IdCategoria = l.IdCategoria
    WHERE l.ValorDC = 'C' AND l.IdCategoria IN (1, 2, 3, 9)
    GROUP BY YEAR(l.DtMovimento), cat.Descricao
    ORDER BY ano, categoria
''')
with eng.connect() as c:
    frap = pd.read_sql(q_frap, c)

# Rotulos curtos por categoria
mapa_cat = {
    "Crédito de Ordem Bancária do SIGEF": "Ordem Bancária (SIGEF)",
    "Crédito por boleto/guia (Exe_Boleto)": "Boleto / Guia",
    "TED, PIX, transferência avulsa": "TED / PIX / transf.",
    "Histórico não classificado – investigar": "Outros",
}
frap["cat_curta"] = frap["categoria"].map(lambda s: mapa_cat.get(s, s))
piv = frap.pivot_table(index="ano", columns="cat_curta", values="total", aggfunc="sum", fill_value=0)
ordem = [c for c in ["Ordem Bancária (SIGEF)", "Boleto / Guia", "TED / PIX / transf.", "Outros"] if c in piv.columns]
piv = piv[ordem]
total_periodo = float(frap["total"].sum())
ano_max = int(frap["ano"].max())
print(piv.round(2)); print("Total periodo:", fmt_brl(total_periodo))

NUM["frap"] = {
    "total_periodo": round(total_periodo, 2),
    "total_periodo_fmt": fmt_brl(total_periodo),
    "ano_inicio": int(frap["ano"].min()), "ano_fim": ano_max,
    "por_ano": {int(a): round(float(v), 2) for a, v in piv.sum(axis=1).items()},
    "por_categoria": {k: round(float(v), 2) for k, v in frap.groupby("cat_curta")["total"].sum().items()},
}

cat_curta  Ordem Bancária (SIGEF)  Boleto / Guia  TED / PIX / transf.
ano                                                                  
2021                    406656.83      309608.23             21675.27
2022                    563985.99      556040.46            339283.55
2023                    673444.76      537796.63              6080.63
2024                    818264.84      750615.21              4475.21
2025                   1772078.92      933993.78             90698.38
2026                    467233.32      247131.80             57393.89
Total periodo: R$ 8.556.457,70


In [3]:
# Grafico 1: arrecadacao por ano (barras empilhadas por forma de pagamento)
cores = [AZUL, TEAL, OURO, CINZA]
fig, ax = plt.subplots(figsize=(9, 5))
base = pd.Series(0.0, index=piv.index)
for i, col in enumerate(piv.columns):
    ax.bar(piv.index.astype(str), piv[col], bottom=base, label=col, color=cores[i % len(cores)])
    base += piv[col]
for x, ano in enumerate(piv.index):
    tot = piv.loc[ano].sum()
    rotulo = fmt_mi(tot) + ("\n(parcial)" if ano == ano_max else "")
    ax.text(x, tot, rotulo, ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.set_title("Arrecadação do FRAP por ano e forma de pagamento")
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v/1_000_000:.1f}M"))
ax.set_ylabel("R$ (milhões)")
ax.set_ylim(0, base.max() * 1.18)
ax.legend(loc="upper left", fontsize=10, frameon=False)
salvar(fig, "frap_por_ano.png")

salvo: scripts\apresentacao\assets\frap_por_ano.png


In [4]:
# Grafico 2: composicao por forma de pagamento (todo o periodo)
serie = frap.groupby("cat_curta")["total"].sum().sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8.5, 4))
barras = ax.barh(serie.index, serie.values, color=[CINZA, OURO, TEAL, AZUL][: len(serie)])
for b, v in zip(barras, serie.values):
    pct = v / serie.sum() * 100
    ax.text(v, b.get_y() + b.get_height() / 2, f"  {fmt_mi(v)} ({pct:.0f}%)", va="center", fontsize=11, fontweight="bold")
ax.set_title("FRAP — composição da arrecadação por forma de pagamento")
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v/1_000_000:.1f}M"))
ax.set_xlim(0, serie.max() * 1.25)
ax.set_xlabel("R$ (milhões)")
salvar(fig, "frap_por_forma.png")

salvo: scripts\apresentacao\assets\frap_por_forma.png


In [5]:
# Piloto Pix / Cartao de credito (canal digital novo) - CSV de solicitacoes via API BB
pix = pd.read_csv(REPO / "scripts" / "docs" / "info_pix_cartao_042026.csv")
pix["tipo"] = pix["CodigoTipoPagamento"].map({"P": "Pix", "C": "Cartão de crédito"})
g = pix.groupby("tipo").agg(qtd=("IdDebitoPagamento", "size"), valor=("ValorSolicitacaoPagamento", "sum"))
# Confirmados = solicitacoes com data de pagamento efetivada
conf = pix[pix["DataHoraPagamento"].notna()]
gc = conf.groupby("tipo").agg(qtd=("IdDebitoPagamento", "size"), valor=("ValorSolicitacaoPagamento", "sum"))
valor_conf = float(conf["ValorSolicitacaoPagamento"].sum())
NUM["pix_cartao"] = {
    "qtd_solicitacoes": int(len(pix)),
    "debitos_distintos": int(pix["IdDebito"].nunique()),
    "valor_solicitado": round(float(pix["ValorSolicitacaoPagamento"].sum()), 2),
    "valor_solicitado_fmt": fmt_brl(float(pix["ValorSolicitacaoPagamento"].sum())),
    "confirmados": int(len(conf)),
    "valor_confirmado": round(valor_conf, 2),
    "valor_confirmado_fmt": fmt_brl(valor_conf),
    "confirmados_por_tipo": {t: {"qtd": int(r.qtd), "valor": round(float(r.valor), 2)} for t, r in gc.iterrows()},
    "por_tipo": {t: {"qtd": int(r.qtd), "valor": round(float(r.valor), 2)} for t, r in g.iterrows()},
    "periodo": [str(pix["DataHoraSolicitacao"].min())[:10], str(pix["DataHoraSolicitacao"].max())[:10]],
}
print(g); print("confirmados:", len(conf), "de", len(pix), "->", fmt_brl(valor_conf))

fig, ax = plt.subplots(figsize=(7.5, 4.2))
barras = ax.bar(g.index, g["valor"], color=[VERMELHO, AZUL], width=0.55)
for b, (_, r) in zip(barras, g.iterrows()):
    ax.text(b.get_x() + b.get_width() / 2, r.valor, f"{fmt_brl(r.valor, 0)}\n{int(r.qtd)} solicitações",
            ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.set_title("Piloto de arrecadação digital — Pix e Cartão (fev–abr/2026)")
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v/1000:.0f}k"))
ax.set_ylabel("R$ solicitado")
ax.set_ylim(0, g["valor"].max() * 1.30)
salvar(fig, "frap_pix_cartao.png")

                   qtd      valor
tipo                             
Cartão de crédito   19  231063.35
Pix                 21  235948.92
confirmados: 1 de 40 -> R$ 1.481,87


salvo: scripts\apresentacao\assets\frap_pix_cartao.png


## 2. Caso Nereu

In [6]:
f_nereu = DOCS / "debitos_nereu_planilha_final.xlsx"
totais = pd.read_excel(f_nereu, "Totais")
proc = pd.read_excel(f_nereu, "Processos")
deb = pd.read_excel(f_nereu, "Débitos")
conc = pd.read_excel(f_nereu, "Conciliações FRAP")

def cat_prescr(s):
    s = str(s)
    if s.startswith("ok"): return "Prazo seguro"
    if s.startswith("risco"): return "Risco (< 1 ano)"
    if s.startswith("PRESCRITO"): return "Prescrito"
    if "imprescr" in s: return "Imprescritível"
    return "Sem dado"
proc["sit"] = proc["risco_prescricao"].map(cat_prescr)
presc = proc["sit"].value_counts()

NUM["nereu"] = {
    "qtd_debitos": int(totais.iloc[0]["quantidade"]),
    "valor_total_fmt": str(totais.iloc[0]["valor_atualizado_R$"]).strip(),
    "qtd_notificados": int(totais.iloc[1]["quantidade"]),
    "valor_notificado_fmt": str(totais.iloc[1]["valor_atualizado_R$"]).strip(),
    "qtd_processos": int(len(proc)),
    "prescricao": {k: int(v) for k, v in presc.items()},
    "tipos_debito": {k: int(v) for k, v in deb["tipo_debito"].value_counts().items()},
    "frap_parcelas": int(len(conc)),
    "frap_pago": round(float(conc["valor_parcela"].sum()), 2),
    "frap_pago_fmt": fmt_brl(float(conc["valor_parcela"].sum())),
}
print(json.dumps(NUM["nereu"], ensure_ascii=False, indent=2))

{
  "qtd_debitos": 594,
  "valor_total_fmt": "R$ 4.871.938,09",
  "qtd_notificados": 41,
  "valor_notificado_fmt": "R$ 415.405,29",
  "qtd_processos": 591,
  "prescricao": {
    "Prazo seguro": 392,
    "Risco (< 1 ano)": 161,
    "Prescrito": 36,
    "Sem dado": 2
  },
  "tipos_debito": {
    "Multa": 451,
    "Multa Cominatória": 143
  },
  "frap_parcelas": 10,
  "frap_pago": 12435.15,
  "frap_pago_fmt": "R$ 12.435,15"
}


In [7]:
# Grafico 3: risco de prescricao dos processos do Nereu
ordem = ["Prazo seguro", "Risco (< 1 ano)", "Prescrito"]
vals = [int(presc.get(k, 0)) for k in ordem]
cores_p = [TEAL, OURO, VERMELHO]
fig, ax = plt.subplots(figsize=(7.5, 4.2))
barras = ax.bar(ordem, vals, color=cores_p, width=0.6)
tot = sum(vals)
for b, v in zip(barras, vals):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v}\n({v/tot*100:.0f}%)", ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.set_title("Caso Nereu — risco de prescrição (591 processos)")
ax.set_ylabel("Processos")
ax.set_ylim(0, max(vals) * 1.2)
salvar(fig, "nereu_prescricao.png")

salvo: scripts\apresentacao\assets\nereu_prescricao.png


## 3. Acervo do CGAD (BdDIP)

In [8]:
ATIVO = "(Cancelado IS NULL OR Cancelado = 0)"
with eng.connect() as c:
    # ATENCAO: DataCumprimento e a DATA-LIMITE (prazo) de cumprimento, NAO a comprovacao de
    # cumprimento efetivo (ha datas no futuro; descricoes do tipo "prazo de 18 meses"). A base
    # do CGAD nao registra se a obrigacao foi de fato cumprida -> medimos situacao de prazo.
    obr = pd.read_sql(text(f"""
        SELECT COUNT(*) n,
               SUM(CASE WHEN DataCumprimento IS NOT NULL THEN 1 ELSE 0 END) com_prazo,
               SUM(CASE WHEN DataCumprimento <= CAST(GETDATE() AS date) THEN 1 ELSE 0 END) prazo_vencido,
               SUM(CASE WHEN DataCumprimento >  CAST(GETDATE() AS date) THEN 1 ELSE 0 END) prazo_vincendo,
               SUM(CASE WHEN TemMultaCominatoria = 1 THEN 1 ELSE 0 END) com_multa,
               SUM(CASE WHEN TemMultaCominatoria = 1 THEN ValorMultaCominatoria ELSE 0 END) valor_multa
        FROM Obrigacao WHERE {ATIVO}"""), c).iloc[0]
    rec = pd.read_sql(text(f"SELECT COUNT(*) n FROM Recomendacao WHERE {ATIVO}"), c).iloc[0]
    classif = pd.read_sql(text("SELECT Classificacao, COUNT(*) n FROM ClassificacaoDecisao GROUP BY Classificacao ORDER BY n DESC"), c)
    orgaos = pd.read_sql(text(f"SELECT IdOrgaoResponsavel idorg, OrgaoResponsavel org FROM Obrigacao WHERE {ATIVO}"), c)
    # Por ano em que o prazo (data-limite) de cumprimento vence
    prazo_ano = pd.read_sql(text(f"SELECT YEAR(DataCumprimento) ano, COUNT(*) n FROM Obrigacao WHERE {ATIVO} AND DataCumprimento IS NOT NULL GROUP BY YEAR(DataCumprimento) ORDER BY ano"), c)

# Resolucao de entidade por IdOrgaoResponsavel (chave autoritativa). Os ids 308 e 521 sao
# registros duplicados do mesmo IPERN estadual; ha dezenas de RPPS municipais distintos.
def _municipal(u):
    return ("MUNIC" in u) or any(k in u for k in
        ["NATALPREV", "MACAUPREV", "IPREVSAPP", "EXTREMOZ", "IPBS", "IPLAP", "IPSAT", "BOA SA", "MACAÍBA", "MOSSOR"])
def classifica_ente(idorg, org):
    u = str(org).upper()
    if idorg in (308.0, 521.0):
        return "IPERN (estadual)"
    if "PREVID" in u:
        estadual = any(k in u for k in ["ESTADO DO R", "SERVIDORES DO RN", "SOCIAL DO RIO GRANDE DO NORTE",
                                        "SERVIDORES DO ESTADO DO RIO GRANDE", "SERVIDORES DO RIO GRANDE DO NORTE"])
        if ("IPERN" in u or estadual) and not _municipal(u):
            return "IPERN (estadual)"
        return "Outros RPPS (municipais)"
    if str(org).strip().lower() in ("desconhecido", "nan", "none", ""):
        return "Não identificado"
    return "Demais órgãos"

orgaos["ente"] = orgaos.apply(lambda r: classifica_ente(r.idorg, r.org), axis=1)
cat_ente = orgaos["ente"].value_counts()
ipern_n = int(cat_ente.get("IPERN (estadual)", 0))
prev_total = ipern_n + int(cat_ente.get("Outros RPPS (municipais)", 0))
sem_prazo = int(obr.n) - int(obr.com_prazo)

NUM["cgad"] = {
    "obrigacoes": int(obr.n),
    "obrigacoes_com_prazo": int(obr.com_prazo),
    "obrigacoes_prazo_vencido": int(obr.prazo_vencido),
    "obrigacoes_prazo_vincendo": int(obr.prazo_vincendo),
    "obrigacoes_sem_prazo": sem_prazo,
    "obrigacoes_com_multa": int(obr.com_multa), "valor_multa_cominatoria_fmt": fmt_brl(float(obr.valor_multa)),
    "recomendacoes": int(rec.n),
    "classificacao_decisoes": {r.Classificacao: int(r.n) for _, r in classif.iterrows()},
    "ipern_obrigacoes": ipern_n, "ipern_pct": round(ipern_n / int(obr.n) * 100, 1),
    "previdencia_total": prev_total, "previdencia_pct": round(prev_total / int(obr.n) * 100, 1),
    "obrigacoes_por_ente": {k: int(v) for k, v in cat_ente.items()},
    "obs": "DataCumprimento e a data-limite (prazo), nao a comprovacao de cumprimento.",
}
print(json.dumps(NUM["cgad"], ensure_ascii=False, indent=2))

{
  "obrigacoes": 1333,
  "obrigacoes_com_prazo": 684,
  "obrigacoes_prazo_vencido": 679,
  "obrigacoes_prazo_vincendo": 5,
  "obrigacoes_sem_prazo": 649,
  "obrigacoes_com_multa": 634,
  "valor_multa_cominatoria_fmt": "R$ 314.285,16",
  "recomendacoes": 899,
  "classificacao_decisoes": {
    "DETERMINACAO": 198,
    "OUTROS": 177
  },
  "ipern_obrigacoes": 446,
  "ipern_pct": 33.5,
  "previdencia_total": 643,
  "previdencia_pct": 48.2,
  "obrigacoes_por_ente": {
    "Demais órgãos": 649,
    "IPERN (estadual)": 446,
    "Outros RPPS (municipais)": 197,
    "Não identificado": 41
  },
  "obs": "DataCumprimento e a data-limite (prazo), nao a comprovacao de cumprimento."
}


In [9]:
# Grafico 4: obrigacoes por tipo de ente (IPERN estadual em destaque)
ordem_e = ["IPERN (estadual)", "Outros RPPS (municipais)", "Demais órgãos", "Não identificado"]
s = pd.Series({k: int(cat_ente.get(k, 0)) for k in ordem_e})
s = s[s > 0].iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 4.4))
cores_o = [AZUL if "IPERN" in n else (TEAL if "RPPS" in n else CINZA) for n in s.index]
barras = ax.barh(s.index, s.values, color=cores_o)
for b, v in zip(barras, s.values):
    ax.text(v, b.get_y() + b.get_height() / 2, f"  {v}  ({v/int(obr.n)*100:.0f}%)", va="center", fontsize=11, fontweight="bold")
ax.set_title("CGAD — obrigações por tipo de ente responsável")
ax.set_xlabel("Obrigações")
ax.set_xlim(0, s.max() * 1.18)
salvar(fig, "cgad_orgaos.png")

salvo: scripts\apresentacao\assets\cgad_orgaos.png


In [10]:
# Grafico 5: situacao de prazo das obrigacoes (a base nao mede cumprimento efetivo)
prazo = prazo_ano[prazo_ano["ano"].between(2023, 2027)].copy()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.3), gridspec_kw={"width_ratios": [1.5, 1]})
# (a) por ano-limite de cumprimento
barras = ax1.bar(prazo["ano"].astype(int).astype(str), prazo["n"], color=AZUL)
for b, (_, r) in zip(barras, prazo.iterrows()):
    ax1.text(b.get_x() + b.get_width() / 2, r.n, f"{int(r.n)}", ha="center", va="bottom", fontsize=10, fontweight="bold")
ax1.set_title("Obrigações por ano-limite de cumprimento")
ax1.set_ylabel("Obrigações")
ax1.set_ylim(0, prazo["n"].max() * 1.2)
# (b) situacao do prazo (relativa a hoje)
venc = int(obr.prazo_vencido); vinc = int(obr.prazo_vincendo)
dados = [("Prazo vencido", venc, VERMELHO), ("No prazo", vinc, TEAL), ("Sem prazo definido", sem_prazo, CINZA)]
dados = [d for d in dados if d[1] > 0]
ax2.pie([d[1] for d in dados], labels=[d[0] for d in dados], colors=[d[2] for d in dados],
        autopct=lambda p: f"{p:.0f}%", startangle=90, wedgeprops={"width": 0.45}, pctdistance=0.78)
ax2.set_title(f"Situação do prazo (de {int(obr.n)})")
salvar(fig, "cgad_cumprimento.png")

salvo: scripts\apresentacao\assets\cgad_cumprimento.png


In [11]:
# numeros.json consolidado
out = REPO / "scripts" / "apresentacao" / "numeros.json"
out.write_text(json.dumps(NUM, ensure_ascii=False, indent=2), encoding="utf-8")
print("escrito:", out.relative_to(REPO))
print(json.dumps(NUM, ensure_ascii=False, indent=2))

escrito: scripts\apresentacao\numeros.json
{
  "frap": {
    "total_periodo": 8556457.7,
    "total_periodo_fmt": "R$ 8.556.457,70",
    "ano_inicio": 2021,
    "ano_fim": 2026,
    "por_ano": {
      "2021": 737940.33,
      "2022": 1459310.0,
      "2023": 1217322.02,
      "2024": 1573355.26,
      "2025": 2796771.08,
      "2026": 771759.01
    },
    "por_categoria": {
      "Boleto / Guia": 3335186.11,
      "Ordem Bancária (SIGEF)": 4701664.66,
      "TED / PIX / transf.": 519606.93
    }
  },
  "pix_cartao": {
    "qtd_solicitacoes": 40,
    "debitos_distintos": 21,
    "valor_solicitado": 467012.27,
    "valor_solicitado_fmt": "R$ 467.012,27",
    "confirmados": 1,
    "valor_confirmado": 1481.87,
    "valor_confirmado_fmt": "R$ 1.481,87",
    "confirmados_por_tipo": {
      "Pix": {
        "qtd": 1,
        "valor": 1481.87
      }
    },
    "por_tipo": {
      "Cartão de crédito": {
        "qtd": 19,
        "valor": 231063.35
      },
      "Pix": {
        "qtd": 21,
  